# W12-POST — BEA-QBGF exploratory VAL pilot
**Run All: tests → nine-query allocation-only calibration → final freeze → missing pilot rows. Calibration failure blocks generation.**

Frozen Week12 remains **SFR top10 → FIXED_SENTENCE → Qwen3-8B BASE_FP16**. This is a separate exploratory mechanism, not confirmation or a replacement final method.

Use the **sara_env** kernel. Wait until POST-W9/other GPU work finishes. The original 150 K/L VAL examples × 2 budgets × 3 methods yield **900 logical rows**; absent caches can require up to 900 answer generations plus SFR preprocessing. Completed compatible rows resume automatically. No confirmation or QASPER TEST is loaded. Expansion/LOO are disabled by default.

In [1]:
from pathlib import Path
import sys, json, subprocess
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
while not (ROOT/'src/week12/algorithm.py').exists():
    if ROOT == ROOT.parent: raise FileNotFoundError('Open this notebook inside CS787-RAG-Project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
RUN_PILOT = True
AUTO_RUN_MISSING = True
REUSE_COMPATIBLE_CACHE = True
OVERWRITE_EXISTING = False
FORCE_REGENERATE = False
RESUME = True
RUN_EXPANDED = False
RUN_ATTRIBUTION = False
RUN_TESTS = True
if OVERWRITE_EXISTING or FORCE_REGENERATE or not RESUME:
    raise ValueError('Preserve results. Choose a new output version for deliberate regeneration.')
from src.post_w12.bea_qbgf.protocol import load, output, freeze, expanded
from src.post_w12.bea_qbgf.runner import plan, prepare, generate
from src.post_w12.bea_qbgf.analysis import analyze
cfg = load(ROOT/'W12-POST/config.json')
cfg['cache']['reuse_compatible'] = REUSE_COMPATIBLE_CACHE
RESULT_ROOT = output(cfg)
print('POST_W12_EXPLORATORY — pilot only; final W12 policy unchanged')


POST_W12_EXPLORATORY — pilot only; final W12 policy unchanged


## 1. Model-free audit, preregistration and automatic missing-cache flag
This verifies original source/config/split/freeze identities, recovers the exact K/L cohort and preregisters calibration before any answer outcomes. The final BEA identity is frozen only after allocation-only calibration passes. Original caches are only counted here; their contents are accepted later only after exact current source/pool/evidence/prompt verification. Nothing is restored or written into Week12 directories.

In [2]:
execution_plan = plan(cfg)
RUN_MISSING = execution_plan['automatic_run_missing'] if AUTO_RUN_MISSING else cfg['cache']['run_missing']
print(json.dumps(execution_plan, indent=2))
print('AUTO RUN_MISSING =', RUN_MISSING)
if RUN_MISSING:
    print('Compatible completed rows are retained. Missing/ambiguous legacy outputs will be recreated using unchanged frozen methods.')
else:
    print('All pilot rows are saved and compatible: no model inference is needed.')
for name in ['AUDIT_BEFORE_RUN.md', 'AUDIT_RUNTIME.md']:
    if (RESULT_ROOT/name).exists(): display(Markdown((RESULT_ROOT/name).read_text()))


{
  "label": "POST_W12_EXPLORATORY",
  "stage": "pilot",
  "queries": 150,
  "logical_rows": 900,
  "compatible_local_rows": 900,
  "missing_or_unverified": 0,
  "prepared_selection_hits": 900,
  "automatic_run_missing": false,
  "physical_generation_upper_bound": 0,
  "legacy_inventory": {
    "validation/C/rows": {
      "exists": true,
      "identity_present": false,
      "files": 0
    },
    "validation/C/prepared": {
      "exists": true,
      "identity_present": false,
      "files": 0
    },
    "generation": {
      "exists": true,
      "identity_present": false,
      "files": 0
    }
  },
  "legacy_rows_not_yet_verified": true,
  "identity_hash": "7b6623f9bbc2ba1e0800d21485ad61b373f6d1f1e49beb401b53e8ae53185d7a",
  "note": "Legacy presence alone is not a hit. Exact current pool/evidence/prompt is verified after preparation. Entirely absent caches trigger all missing pilot rows, never historical reruns or automatic expansion."
}
AUTO RUN_MISSING = False
All pilot rows are

# BEA-QBGF runtime preflight

POST_W12_EXPLORATORY; VAL only; final W12 policy unchanged.

```json
{
  "label": "POST_W12_EXPLORATORY",
  "stage": "pilot",
  "queries": 150,
  "logical_rows": 900,
  "compatible_local_rows": 900,
  "missing_or_unverified": 0,
  "prepared_selection_hits": 900,
  "automatic_run_missing": false,
  "physical_generation_upper_bound": 0,
  "legacy_inventory": {
    "validation/C/rows": {
      "exists": true,
      "identity_present": false,
      "files": 0
    },
    "validation/C/prepared": {
      "exists": true,
      "identity_present": false,
      "files": 0
    },
    "generation": {
      "exists": true,
      "identity_present": false,
      "files": 0
    }
  },
  "legacy_rows_not_yet_verified": true,
  "identity_hash": "7b6623f9bbc2ba1e0800d21485ad61b373f6d1f1e49beb401b53e8ae53185d7a",
  "note": "Legacy presence alone is not a hit. Exact current pool/evidence/prompt is verified after preparation. Entirely absent caches trigger all missing pilot rows, never historical reruns or automatic expansion."
}
```

Canonical definitions: src/week12/algorithm.py (FIXED_SENTENCE/QBGF_FULL/serializer), src/week12/laboratory.py (SFR top10), src/week11/hierarchy.py, src/week12/reader.py. Models, revisions, precision, tokenizer, prompt/parser/source hashes and exact cohort are in BEA_QBGF_V1_FREEZE.json and PILOT_COHORT.json. Missing embedding/hierarchy caches are reconstructed only under this new output root. Frozen SQLite sources are read-only.


## 2. Tests before allocation calibration
Run new BEA tests and unchanged Week12 tests. Calibration also enforces a compatible successful test receipt. No answer generation occurs.

In [3]:
if RUN_TESTS:
    from src.post_w12.bea_qbgf.calibration import tests
    tests(cfg)
else:
    print('Calibration will enforce tests before running.')


## 2b. Allocation-only calibration and final freeze
Nine exact pilot VAL IDs; no reader generation or F1. A global preregistered threshold grid uses only allocation statistics. NEEDS_REVIEW blocks the pilot. Existing completed calibration is reused. FULL eligibility after two positive parent selections is a design assumption.


In [4]:
if RUN_PILOT:
    from src.post_w12.bea_qbgf.calibration import run as calibrate
    calibration = calibrate(cfg)
    display(Markdown((RESULT_ROOT/'calibration/ALLOCATION_ONLY_REPORT.md').read_text()))
    parent, split, cohort, frozen = freeze(cfg)
    print('Frozen calibrated parameters:', frozen['identity']['parameters'])
    execution_plan = plan(cfg)
    RUN_MISSING = execution_plan['automatic_run_missing'] if AUTO_RUN_MISSING else cfg['cache']['run_missing']


# Allocation-only pre-pilot calibration

Nine frozen VAL pilot queries (three per dataset); both budgets. No Qwen generation or answer-quality scoring. Native VAL adapters parse their existing records, but evaluation payloads are discarded before allocation. The threshold rule was registered before observing these traces. FULL eligibility remains a two-positive-sentence design assumption.

|    tau |   n |   threshold_stop_fraction |   budget_stop_fraction |   mean_utilization |   near_full_fraction |   empty_fraction |   coverage_retention | eligible   |
|-------:|----:|--------------------------:|-----------------------:|-------------------:|---------------------:|-----------------:|---------------------:|:-----------|
| 0.001  |  18 |                 0         |               1        |          0.983398  |             0.888889 |                0 |             1        | False      |
| 0.0025 |  18 |                 0.0555556 |               0.944444 |          0.983073  |             0.888889 |                0 |             1        | False      |
| 0.005  |  18 |                 0.166667  |               0.833333 |          0.959147  |             0.777778 |                0 |             1        | False      |
| 0.01   |  18 |                 0.722222  |               0.277778 |          0.67117   |             0.222222 |                0 |             0.996054 | True       |
| 0.02   |  18 |                 1         |               0        |          0.139431  |             0        |                0 |             0.960694 | True       |
| 0.04   |  18 |                 1         |               0        |          0.0298394 |             0        |                0 |             0.824658 | False      |

Status: **READY**; selected global threshold: 0.01.

FULL counts are candidate-step events, not distinct sources. `full_eligible_query` counts queries with any eligible candidate.

| dataset   |   budget |    tau |   full_units |   full_eligible_query |   full_chosen |   full_rejected_cost |   full_rejected_low_gain |   full_rejected_threshold |   full_lost_competition |   full_blocked_one_sentence |   full_considered |
|:----------|---------:|-------:|-------------:|----------------------:|--------------:|---------------------:|-------------------------:|--------------------------:|------------------------:|----------------------------:|------------------:|
| HotpotQA  |      256 | 0.001  |            1 |                     3 |             1 |                   99 |                        0 |                         0 |                      40 |                         329 |               140 |
| HotpotQA  |      256 | 0.0025 |            1 |                     3 |             1 |                   99 |                        0 |                         0 |                      40 |                         329 |               140 |
| HotpotQA  |      256 | 0.005  |            1 |                     3 |             1 |                   99 |                        0 |                         1 |                      39 |                         329 |               140 |
| HotpotQA  |      256 | 0.01   |            1 |                     3 |             1 |                   71 |                        0 |                        25 |                      14 |                         297 |               111 |
| HotpotQA  |      256 | 0.02   |            1 |                     2 |             1 |                    5 |                        0 |                         9 |                       3 |                         121 |                18 |
| HotpotQA  |      256 | 0.04   |            0 |                     0 |             0 |                    0 |                        0 |                         0 |                       0 |                          27 |                 0 |
| HotpotQA  |     1024 | 0.001  |           17 |                     3 |            17 |                   45 |                        0 |                         0 |                     609 |                         429 |               671 |
| HotpotQA  |     1024 | 0.0025 |           17 |                     3 |            17 |                   39 |                        0 |                        26 |                     583 |                         429 |               665 |
| HotpotQA  |     1024 | 0.005  |           12 |                     3 |            12 |                    7 |                        0 |                       280 |                     295 |                         425 |               594 |
| HotpotQA  |     1024 | 0.01   |            1 |                     3 |             1 |                    0 |                        0 |                       145 |                      15 |                         327 |               161 |
| HotpotQA  |     1024 | 0.02   |            1 |                     2 |             1 |                    0 |                        0 |                        14 |                       3 |                         121 |                18 |
| HotpotQA  |     1024 | 0.04   |            0 |                     0 |             0 |                    0 |                        0 |                         0 |                       0 |                          27 |                 0 |
| QASPER    |      256 | 0.001  |            0 |                     3 |             0 |                  148 |                        0 |                         0 |                       1 |                         211 |               149 |
| QASPER    |      256 | 0.0025 |            0 |                     3 |             0 |                  148 |                        0 |                         0 |                       1 |                         211 |               149 |
| QASPER    |      256 | 0.005  |            0 |                     3 |             0 |                  148 |                        0 |                         0 |                       1 |                         211 |               149 |
| QASPER    |      256 | 0.01   |            0 |                     3 |             0 |                  131 |                        0 |                         1 |                       0 |                         199 |               132 |
| QASPER    |      256 | 0.02   |            0 |                     2 |             0 |                   15 |                        0 |                         0 |                       0 |                          33 |                15 |
| QASPER    |      256 | 0.04   |            0 |                     1 |             0 |                    1 |                        0 |                         0 |                       0 |                           7 |                 1 |
| QASPER    |     1024 | 0.001  |            0 |                     3 |             0 |                  151 |                        0 |                         0 |                     799 |                         300 |               950 |
| QASPER    |     1024 | 0.0025 |            0 |                     3 |             0 |                  151 |                        0 |                         0 |                     799 |                         300 |               950 |
| QASPER    |     1024 | 0.005  |            0 |                     3 |             0 |                  151 |                        0 |                       416 |                     383 |                         300 |               950 |
| QASPER    |     1024 | 0.01   |            0 |                     3 |             0 |                    0 |                        0 |                       238 |                       0 |                         232 |               238 |
| QASPER    |     1024 | 0.02   |            0 |                     2 |             0 |                    0 |                        0 |                        15 |                       0 |                          33 |                15 |
| QASPER    |     1024 | 0.04   |            0 |                     1 |             0 |                    0 |                        0 |                         1 |                       0 |                           7 |                 1 |
| SciFact   |      256 | 0.001  |            0 |                     3 |             0 |                   89 |                        0 |                         0 |                       0 |                         172 |                89 |
| SciFact   |      256 | 0.0025 |            0 |                     3 |             0 |                   89 |                        0 |                         0 |                       0 |                         172 |                89 |
| SciFact   |      256 | 0.005  |            0 |                     3 |             0 |                   89 |                        0 |                         0 |                       0 |                         172 |                89 |
| SciFact   |      256 | 0.01   |            0 |                     3 |             0 |                   89 |                        0 |                         0 |                       0 |                         172 |                89 |
| SciFact   |      256 | 0.02   |            0 |                     2 |             0 |                    8 |                        0 |                         0 |                       0 |                          39 |                 8 |
| SciFact   |      256 | 0.04   |            0 |                     0 |             0 |                    0 |                        0 |                         0 |                       0 |                           7 |                 0 |
| SciFact   |     1024 | 0.001  |            0 |                     3 |             0 |                  252 |                        0 |                         0 |                     478 |                         367 |               730 |
| SciFact   |     1024 | 0.0025 |            0 |                     3 |             0 |                  252 |                        0 |                         0 |                     478 |                         367 |               730 |
| SciFact   |     1024 | 0.005  |            0 |                     3 |             0 |                  252 |                        0 |                        35 |                     443 |                         367 |               730 |
| SciFact   |     1024 | 0.01   |            0 |                     3 |             0 |                   75 |                        0 |                       406 |                       0 |                         324 |               481 |
| SciFact   |     1024 | 0.02   |            0 |                     2 |             0 |                    0 |                        0 |                         8 |                       0 |                          39 |                 8 |
| SciFact   |     1024 | 0.04   |            0 |                     0 |             0 |                    0 |                        0 |                         0 |                       0 |                           7 |                 0 |


Frozen calibrated parameters: {'eta_cov': 0.5, 'lambda_red': 0.5, 'min_parent_sentences': 2, 'tau_context': 0.05, 'tau_expand': 0.01, 'tau_stop': 0.01}


## 3. Prepare missing contexts — SFR first
Use the unchanged frozen hierarchy, top10 retrieval and token serializer. BEA recomputes global marginal relevance/coverage/redundancy after every selection. FULL becomes a competing action only after source evidence accumulates. Selected source text remains exact. Missing embeddings/hierarchy go into the new extension cache. SFR is released before the reader stage.

In [5]:
if RUN_PILOT:
    prepare(cfg, stage='pilot', run_missing=RUN_MISSING)
else:
    print('Pilot preparation disabled.')


/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Generate only missing rows — Qwen BASE_FP16
Every completed result is saved immediately. Exact compatible historical answers or identical-prompt generation caches are reused; incompatible/ambiguous ones are logged and replaced only in this new experiment. No no-evidence control or additional method is generated. If interrupted, rerun Run All; it continues from durable rows.

In [6]:
if RUN_PILOT:
    pilot_rows = generate(cfg, stage='pilot', run_missing=RUN_MISSING)
    print('Completed logical pilot rows:', len(pilot_rows))
else:
    print('Pilot generation disabled.')


Completed logical pilot rows: 900


## 5. Complete paired analysis and pilot decision
Report all dataset/budget cells, 12 preregistered exploratory paired contrasts, group-aware bootstrap uncertainty, costs and mechanism diagnostics. Partial results cannot be used to choose winners. The pilot gate is fixed globally in config; it does not promote BEA as a final method.

In [7]:
pilot_analysis = None
if RUN_PILOT:
    pilot_analysis = analyze(cfg, stage='pilot')
    for name in ['summary', 'paired_comparisons', 'evidence_behavior']:
        display(pilot_analysis[name])
    display(Markdown((RESULT_ROOT/'PILOT_DECISION.md').read_text()))
    display(Markdown((RESULT_ROOT/'pilot/PILOT_REPORT.md').read_text()))


,label,dataset,budget,method,n_queries,n_groups,metric,quality,EM,accuracy,invalid_format,evidence_tokens,prompt_tokens,generated_tokens,physical_generation_calls_original_receipts,conceptual_lm_calls
0,POST_W12_EXPLORATORY,HotpotQA,256,BEA_QBGF,50,50,f1,0.297642,0.18,NaN,NaN,225.26,280.14,16.92,50,50
1,POST_W12_EXPLORATORY,HotpotQA,256,FIXED_SENTENCE,50,50,f1,0.366109,0.22,NaN,NaN,254.84,309.72,13.54,50,50
2,POST_W12_EXPLORATORY,HotpotQA,256,QBGF_FULL,50,50,f1,0.404225,0.28,NaN,NaN,251.62,306.50,13.60,50,50
3,POST_W12_EXPLORATORY,HotpotQA,1024,BEA_QBGF,50,50,f1,0.344117,0.22,NaN,NaN,284.64,339.52,15.76,28,50
4,POST_W12_EXPLORATORY,HotpotQA,1024,FIXED_SENTENCE,50,50,f1,0.389292,0.24,NaN,NaN,1001.74,1056.62,15.34,50,50
5,POST_W12_EXPLORATORY,HotpotQA,1024,QBGF_FULL,50,50,f1,0.402047,0.26,NaN,NaN,995.26,1050.14,15.28,44,50
6,POST_W12_EXPLORATORY,QASPER,256,BEA_QBGF,50,50,f1,0.186849,0.02,NaN,NaN,245.74,288.32,18.08,50,50
7,POST_W12_EXPLORATORY,QASPER,256,FIXED_SENTENCE,50,50,f1,0.266794,0.02,NaN,NaN,255.06,297.64,18.26,50,50
8,POST_W12_EXPLORATORY,QASPER,256,QBGF_FULL,50,50,f1,0.264119,0.02,NaN,NaN,252.52,295.10,18.74,50,50
9,POST_W12_EXPLORATORY,QASPER,1024,BEA_QBGF,50,50,f1,0.208418,0.02,NaN,NaN,472.78,515.36,19.22,42,50


,label,dataset,budget,left,right,point,ci_low,ci_high,raw_p,resamples,seed,n_queries,n_clusters,metric,test,exploratory_family_size,holm_p
0,POST_W12_EXPLORATORY,HotpotQA,256,BEA_QBGF,FIXED_SENTENCE,-0.068467,-0.181965,0.040067,0.243876,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,1.000000
1,POST_W12_EXPLORATORY,HotpotQA,256,BEA_QBGF,QBGF_FULL,-0.106583,-0.219946,0.000584,0.070893,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,0.567143
2,POST_W12_EXPLORATORY,HotpotQA,1024,BEA_QBGF,FIXED_SENTENCE,-0.045175,-0.137005,0.043336,0.340766,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,1.000000
3,POST_W12_EXPLORATORY,HotpotQA,1024,BEA_QBGF,QBGF_FULL,-0.057930,-0.161530,0.041834,0.274673,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,1.000000
4,POST_W12_EXPLORATORY,QASPER,256,BEA_QBGF,FIXED_SENTENCE,-0.079946,-0.152321,-0.014674,0.028497,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,0.284972
5,POST_W12_EXPLORATORY,QASPER,256,BEA_QBGF,QBGF_FULL,-0.077271,-0.153040,-0.007398,0.049395,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,0.444556
6,POST_W12_EXPLORATORY,QASPER,1024,BEA_QBGF,FIXED_SENTENCE,-0.081185,-0.149839,-0.015581,0.021898,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,0.240876
7,POST_W12_EXPLORATORY,QASPER,1024,BEA_QBGF,QBGF_FULL,-0.087998,-0.151298,-0.030139,0.004300,10000,42,50,50,f1,two-sided paired cluster label-swap randomizat...,12,0.051595
8,POST_W12_EXPLORATORY,SciFact,256,BEA_QBGF,FIXED_SENTENCE,-0.016176,-0.181927,0.148248,0.829017,10000,42,50,50,macro_f1,two-sided paired cluster label-swap randomizat...,12,1.000000
9,POST_W12_EXPLORATORY,SciFact,256,BEA_QBGF,QBGF_FULL,0.081744,-0.066269,0.233638,0.292571,10000,42,50,50,macro_f1,two-sided paired cluster label-swap randomizat...,12,1.000000


,label,dataset,budget,method,n_queries,full_eligible_query,full_considered,full_chosen,full_rejected_cost,full_rejected_low_gain,...,mean_pairwise_redundancy,aspect_coverage,represented_aspects,tokens_per_represented_aspect,fidelity_upgrades,empty_evidence,stopped_with_unused_budget,pairwise_redundancy_defined_queries,full_representation_frequency,upgrade_cell_fraction
0,POST_W12_EXPLORATORY,HotpotQA,256,BEA_QBGF,50,50,1970,14,1108,8,...,0.594611,0.748675,1.64,165.453333,0.28,0.0,0.62,50,0.26,0.26
1,POST_W12_EXPLORATORY,HotpotQA,256,FIXED_SENTENCE,50,0,0,0,0,0,...,0.650451,0.761499,1.64,187.278333,0.00,0.0,0.00,50,0.00,0.00
2,POST_W12_EXPLORATORY,HotpotQA,256,QBGF_FULL,50,0,0,0,0,0,...,0.627003,0.759819,1.64,185.301667,0.00,0.0,0.00,50,0.00,0.00
3,POST_W12_EXPLORATORY,HotpotQA,1024,BEA_QBGF,50,50,2815,25,0,8,...,0.591976,0.755754,1.64,209.426667,0.50,0.0,1.00,50,0.38,0.38
4,POST_W12_EXPLORATORY,HotpotQA,1024,FIXED_SENTENCE,50,0,0,0,0,0,...,0.585496,0.764490,1.64,737.265000,0.00,0.0,0.00,50,0.00,0.00
5,POST_W12_EXPLORATORY,HotpotQA,1024,QBGF_FULL,50,0,0,0,0,0,...,0.573011,0.764431,1.64,732.538333,0.00,0.0,0.00,50,0.12,0.00
6,POST_W12_EXPLORATORY,QASPER,256,BEA_QBGF,50,50,3027,0,3013,0,...,0.584340,0.709078,1.02,243.230000,0.00,0.0,0.28,50,0.00,0.00
7,POST_W12_EXPLORATORY,QASPER,256,FIXED_SENTENCE,50,0,0,0,0,0,...,0.686346,0.722814,1.02,252.510000,0.00,0.0,0.00,50,0.00,0.00
8,POST_W12_EXPLORATORY,QASPER,256,QBGF_FULL,50,0,0,0,0,0,...,0.682180,0.720867,1.02,249.990000,0.00,0.0,0.00,50,0.00,0.00
9,POST_W12_EXPLORATORY,QASPER,1024,BEA_QBGF,50,50,7785,0,231,0,...,0.587056,0.715636,1.02,468.000000,0.00,0.0,0.96,50,0.00,0.00


# BEA-QBGF pilot decision

POST_W12_EXPLORATORY; not confirmation.

```json
{
  "label": "POST_W12_EXPLORATORY",
  "decision": "PROMISING",
  "conditions": {
    "not_clearly_degraded_in_most_cells": true,
    "meaningful_dimension_present": true,
    "different_from_fixed": true,
    "escalation_active": true,
    "not_degenerate_empty": true
  },
  "clearly_degraded_cells": [
    [
      "QASPER",
      256
    ],
    [
      "QASPER",
      1024
    ]
  ],
  "meaningful_dimensions": [
    {
      "dataset": "SciFact",
      "budget": 1024,
      "dimension": "efficiency/coverage/redundancy",
      "comparator": "FIXED_SENTENCE",
      "token_reduction": 0.31809542470853336,
      "redundancy_reduction": 0.08405074373557164,
      "coverage_change": -0.015372023511552091,
      "note": "quality point estimates competitive; not proven statistical noninferiority"
    },
    {
      "dataset": "SciFact",
      "budget": 1024,
      "dimension": "efficiency/coverage/redundancy",
      "comparator": "QBGF_FULL",
      "token_reduction": 0.31677006439453437,
      "redundancy_reduction": 0.04009281454090119,
      "coverage_change": -0.009585728308041719,
      "note": "quality point estimates competitive; not proven statistical noninferiority"
    }
  ],
  "different_context_fraction": 1.0,
  "upgrade_cell_fraction": 0.11333333333333333,
  "empty_cell_fraction": 0.0,
  "interpretation": "Global preregistered exploratory gate; no dataset-specific promotion, no proof of noninferiority, no final-method replacement"
}
```

Expansion requires both PROMISING and a separate manual notebook flag. No final W12 method changes.


# BEA-QBGF pilot — POST_W12_EXPLORATORY

Decision: **PROMISING**. Frozen Week12 remains SFR top10 → FIXED_SENTENCE → Qwen3-8B BASE_FP16. These are reused VAL examples, not fresh confirmation or OOD.

## All quality and cost cells

| label                | dataset   |   budget | method         |   n_queries |   n_groups | metric   |   quality |     EM |   accuracy |   invalid_format |   evidence_tokens |   prompt_tokens |   generated_tokens |   physical_generation_calls_original_receipts |   conceptual_lm_calls |
|:---------------------|:----------|---------:|:---------------|------------:|-----------:|:---------|----------:|-------:|-----------:|-----------------:|------------------:|----------------:|-------------------:|----------------------------------------------:|----------------------:|
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF       |          50 |         50 | f1       |  0.297642 |   0.18 |     nan    |              nan |            225.26 |          280.14 |              16.92 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | FIXED_SENTENCE |          50 |         50 | f1       |  0.366109 |   0.22 |     nan    |              nan |            254.84 |          309.72 |              13.54 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | QBGF_FULL      |          50 |         50 | f1       |  0.404225 |   0.28 |     nan    |              nan |            251.62 |          306.5  |              13.6  |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF       |          50 |         50 | f1       |  0.344117 |   0.22 |     nan    |              nan |            284.64 |          339.52 |              15.76 |                                            28 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | FIXED_SENTENCE |          50 |         50 | f1       |  0.389292 |   0.24 |     nan    |              nan |           1001.74 |         1056.62 |              15.34 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | QBGF_FULL      |          50 |         50 | f1       |  0.402047 |   0.26 |     nan    |              nan |            995.26 |         1050.14 |              15.28 |                                            44 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF       |          50 |         50 | f1       |  0.186849 |   0.02 |     nan    |              nan |            245.74 |          288.32 |              18.08 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |      256 | FIXED_SENTENCE |          50 |         50 | f1       |  0.266794 |   0.02 |     nan    |              nan |            255.06 |          297.64 |              18.26 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |      256 | QBGF_FULL      |          50 |         50 | f1       |  0.264119 |   0.02 |     nan    |              nan |            252.52 |          295.1  |              18.74 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF       |          50 |         50 | f1       |  0.208418 |   0.02 |     nan    |              nan |            472.78 |          515.36 |              19.22 |                                            42 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |     1024 | FIXED_SENTENCE |          50 |         50 | f1       |  0.289603 |   0.02 |     nan    |              nan |           1023.16 |         1065.74 |              20.42 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |     1024 | QBGF_FULL      |          50 |         50 | f1       |  0.296416 |   0.02 |     nan    |              nan |           1020.44 |         1063.02 |              19.44 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF       |          50 |         50 | macro_f1 |  0.612738 | nan    |       0.62 |                0 |            250.08 |          314.04 |               3.84 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |      256 | FIXED_SENTENCE |          50 |         50 | macro_f1 |  0.628915 | nan    |       0.64 |                0 |            252.62 |          316.58 |               3.52 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |      256 | QBGF_FULL      |          50 |         50 | macro_f1 |  0.530994 | nan    |       0.54 |                0 |            250.56 |          314.52 |               3.8  |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF       |          50 |         50 | macro_f1 |  0.643155 | nan    |       0.66 |                0 |            696.02 |          759.98 |               3.68 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | FIXED_SENTENCE |          50 |         50 | macro_f1 |  0.553333 | nan    |       0.64 |                0 |           1020.7  |         1084.66 |               3.16 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | QBGF_FULL      |          50 |         50 | macro_f1 |  0.588105 | nan    |       0.64 |                0 |           1018.72 |         1082.68 |               3.36 |                                            50 |                    50 |

## Paired differences

| label                | dataset   |   budget | left     | right          |      point |     ci_low |      ci_high |      raw_p |   resamples |   seed |   n_queries |   n_clusters | metric   | test                                                                          |   exploratory_family_size |    holm_p |
|:---------------------|:----------|---------:|:---------|:---------------|-----------:|-----------:|-------------:|-----------:|------------:|-------:|------------:|-------------:|:---------|:------------------------------------------------------------------------------|--------------------------:|----------:|
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF | FIXED_SENTENCE | -0.0684668 | -0.181965  |  0.0400673   | 0.243876   |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF | QBGF_FULL      | -0.106583  | -0.219946  |  0.000584362 | 0.0708929  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.567143  |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF | FIXED_SENTENCE | -0.0451749 | -0.137005  |  0.0433362   | 0.340766   |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF | QBGF_FULL      | -0.05793   | -0.16153   |  0.041834    | 0.274673   |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF | FIXED_SENTENCE | -0.0799456 | -0.152321  | -0.0146743   | 0.0284972  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.284972  |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF | QBGF_FULL      | -0.0772708 | -0.15304   | -0.00739775  | 0.0493951  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.444556  |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF | FIXED_SENTENCE | -0.081185  | -0.149839  | -0.0155808   | 0.0218978  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.240876  |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF | QBGF_FULL      | -0.0879982 | -0.151298  | -0.0301388   | 0.00429957 |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.0515948 |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF | FIXED_SENTENCE | -0.0161764 | -0.181927  |  0.148248    | 0.829017   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF | QBGF_FULL      |  0.0817442 | -0.0662691 |  0.233638    | 0.292571   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF | FIXED_SENTENCE |  0.0898216 | -0.0542215 |  0.228745    | 0.293971   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF | QBGF_FULL      |  0.0550504 | -0.0727925 |  0.181957    | 0.429157   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |

## Evidence behavior

| label                | dataset   |   budget | method         |   n_queries |   full_eligible_query |   full_considered |   full_chosen |   full_rejected_cost |   full_rejected_low_gain |   full_rejected_threshold |   full_lost_competition |   full_blocked_one_sentence |   evidence_tokens |   unused_budget |   budget_utilization |   selected_units |   represented_parents |   sentence_units |   full_units |   mean_pairwise_redundancy |   aspect_coverage |   represented_aspects |   tokens_per_represented_aspect |   fidelity_upgrades |   empty_evidence |   stopped_with_unused_budget |   pairwise_redundancy_defined_queries |   full_representation_frequency |   upgrade_cell_fraction |
|:---------------------|:----------|---------:|:---------------|------------:|----------------------:|------------------:|--------------:|---------------------:|-------------------------:|--------------------------:|------------------------:|----------------------------:|------------------:|----------------:|---------------------:|-----------------:|----------------------:|-----------------:|-------------:|---------------------------:|------------------:|----------------------:|--------------------------------:|--------------------:|-----------------:|-----------------------------:|--------------------------------------:|--------------------------------:|------------------------:|
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF       |          50 |                    50 |              1970 |            14 |                 1108 |                        8 |                       594 |                     246 |                        4532 |            225.26 |           30.74 |             0.879922 |            18.16 |                  9.66 |            17.88 |         0.28 |                   0.594611 |          0.748675 |                  1.64 |                         165.453 |                0.28 |                0 |                         0.62 |                                    50 |                            0.26 |                    0.26 |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            254.84 |            1.16 |             0.995469 |             9.8  |                  5.28 |             9.8  |         0    |                   0.650451 |          0.761499 |                  1.64 |                         187.278 |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            251.62 |            4.38 |             0.982891 |            11.74 |                  4.94 |            11.74 |         0    |                   0.627003 |          0.759819 |                  1.64 |                         185.302 |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF       |          50 |                    50 |              2815 |            25 |                    0 |                        8 |                      2434 |                     348 |                        4802 |            284.64 |          739.36 |             0.277969 |            20.02 |                  9.72 |            19.52 |         0.5  |                   0.591976 |          0.755754 |                  1.64 |                         209.427 |                0.5  |                0 |                         1    |                                    50 |                            0.38 |                    0.38 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1001.74 |           22.26 |             0.978262 |            36.06 |                  9.62 |            36.06 |         0    |                   0.585496 |          0.76449  |                  1.64 |                         737.265 |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            995.26 |           28.74 |             0.971934 |            38.28 |                  9.24 |            38.14 |         0.14 |                   0.573011 |          0.764431 |                  1.64 |                         732.538 |                0    |                0 |                         0    |                                    50 |                            0.12 |                    0    |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF       |          50 |                    50 |              3027 |             0 |                 3013 |                        0 |                        12 |                       2 |                        3240 |            245.74 |           10.26 |             0.959922 |            20.3  |                  8.78 |            20.3  |         0    |                   0.58434  |          0.709078 |                  1.02 |                         243.23  |                0    |                0 |                         0.28 |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |      256 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            255.06 |            0.94 |             0.996328 |             9.34 |                  5.7  |             9.34 |         0    |                   0.686346 |          0.722814 |                  1.02 |                         252.51  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |      256 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            252.52 |            3.48 |             0.986406 |            10.22 |                  5.82 |            10.22 |         0    |                   0.68218  |          0.720867 |                  1.02 |                         249.99  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF       |          50 |                    50 |              7785 |             0 |                  231 |                        0 |                      7027 |                     527 |                        3946 |            472.78 |          551.22 |             0.461699 |            31.9  |                  9.3  |            31.9  |         0    |                   0.587056 |          0.715636 |                  1.02 |                         468     |                0    |                0 |                         0.96 |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |     1024 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1023.16 |            0.84 |             0.99918  |            32.2  |                  9.62 |            32.2  |         0    |                   0.66966  |          0.722814 |                  1.02 |                        1012.92  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |     1024 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1020.44 |            3.56 |             0.996523 |            24.5  |                  8.56 |            23.1  |         1.4  |                   0.639274 |          0.722404 |                  1.02 |                        1010.26  |                0    |                0 |                         0    |                                    50 |                            0.86 |                    0    |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF       |          50 |                    50 |              1462 |             0 |                 1401 |                        0 |                        24 |                      37 |                        2703 |            250.08 |            5.92 |             0.976875 |            15.46 |                  8.16 |            15.46 |         0    |                   0.629506 |          0.777207 |                  1.2  |                         225.12  |                0    |                0 |                         0.04 |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |      256 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            252.62 |            3.38 |             0.986797 |             6.56 |                  3.22 |             6.56 |         0    |                   0.750613 |          0.802011 |                  1.2  |                         227.44  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |      256 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            250.56 |            5.44 |             0.97875  |             8.5  |                  3.36 |             8.5  |         0    |                   0.721572 |          0.785962 |                  1.2  |                         225.29  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF       |          50 |                    50 |              8027 |             2 |                 1281 |                        0 |                      6490 |                     254 |                        4871 |            696.02 |          327.98 |             0.679707 |            33.98 |                  9.8  |            33.94 |         0.04 |                   0.634118 |          0.789252 |                  1.2  |                         624.31  |                0.04 |                0 |                         0.9  |                                    50 |                            0.04 |                    0.04 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1020.7  |            3.3  |             0.996777 |            24.1  |                  8.3  |            24.1  |         0    |                   0.718169 |          0.804624 |                  1.2  |                         918.67  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |     1024 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1018.72 |            5.28 |             0.994844 |            27.7  |                  7.02 |            27.22 |         0.48 |                   0.674211 |          0.798838 |                  1.2  |                         916.89  |                0    |                0 |                         0    |                                    50 |                            0.42 |                    0    |

## Questions answered

1. **Better than QBGF?** See BEA−QBGF paired differences for every dataset/budget; mixed cells do not establish universal superiority.

2. **Competitive with FIXED_SENTENCE?** See BEA−FIXED paired intervals. Point competitiveness in the pilot gate is not a formal noninferiority result.

3. **Less redundancy?** Compare mean_pairwise_redundancy with its defined-query counts; coverage of FULL uses its constituent sentences for comparability.

4. **More query-aspect coverage?** Compare aspect_coverage under identical query-only aspects. Cosine coverage is a surrogate, not verified fact coverage.

5. **Stops with unused tokens?** See unused_budget, stopped_with_unused_budget and saved per-step stop reasons.

6. **Fidelity escalation active?** Upgrade-cell fraction: 0.1133; inspect source accumulation and FULL actions in cache/selections.

7. **Does expansion itself improve answers?** Unresolved: frequency is measured, but there is no separate BEA-without-expansion primary arm. Do not infer causal benefit from activation or between-method scores.

8. **Gains at 256 or 1024?** The complete paired table reports both budgets; do not choose a budget after outcomes.

9. **Dataset dependence?** Report HotpotQA/QASPER answer F1 and SciFact aggregate macro-F1 separately; never pool them.

10. **Top-down relevance-mass hypothesis supported?** Mechanism diagnostics can support or weaken its plausibility, but without new LOO this remains unresolved as a claim about neutral downstream evidence. Relevance/coverage/redundancy proxies are not downstream utility labels.

## Scope and costs

All three methods share exact inputs, frozen serializer, reader, prompt/parser, precision and decoding. Physical cache reuse is recorded per row; every system still costs one conceptual answer call. Embedding lookup/physical/original preprocessing costs and query/aspect computation are recorded in cache/problems; they are distinct from generation. No end-to-end latency or serving-throughput claim is made. No gold/LOO/answer-score fields enter allocation. Novelty is diagnostic only, not a duplicate reward. FULL utility sums sequentially recomputed positive marginal sentence surrogates, updating redundancy and coverage after hypothetical additions; it remains a surrogate rather than learned downstream contextual utility.

Optional attribution is not run automatically. Neutral LOO means zero score change under removal, not useless or unobserved evidence. Expanded VAL is blocked unless the pilot passes and the user enables it; it is not independent confirmation. Historical W12 confirmation/final freeze remain unchanged.
Physical call totals in summary are original creation receipts, including pilot rows reused in expanded VAL. For physical calls in the latest invocation, use that stage’s cache_manifest.csv; reused rows are never presented as newly generated.


## 6. Expanded VAL — separate opt-in, blocked unless PROMISING
Default **False**. Keeps the pilot IDs and deterministically adds VAL IDs up to 150/dataset. Reuses compatible pilot/common rows. Does not touch confirmation. This remains exploratory reused VAL, not new confirmation.

In [10]:
RUN_EXPANDED = True
if RUN_EXPANDED:
    pilot_analysis = analyze(cfg, stage='pilot')
    parent, split, pilot_cohort, frozen = freeze(cfg)
    expanded_cohort = expanded(cfg, split, pilot_cohort, pilot_analysis['decision'])
    expanded_plan = plan(cfg, stage='expanded')
    expanded_missing = expanded_plan['automatic_run_missing']
    prepare(cfg, stage='expanded', run_missing=expanded_missing)
    generate(cfg, stage='expanded', run_missing=expanded_missing)
    expanded_analysis = analyze(cfg, stage='expanded')
    display(expanded_analysis['summary'])
else:
    print('Expanded VAL not launched. Review the complete pilot and opt in separately.')


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 127.83it/s]


BEA preparation 2/450 queries; elapsed=13s; saved selections reused
BEA preparation 3/450 queries; elapsed=19s; saved selections reused
BEA preparation 4/450 queries; elapsed=25s; saved selections reused
BEA preparation 5/450 queries; elapsed=29s; saved selections reused
BEA preparation 7/450 queries; elapsed=33s; saved selections reused
BEA preparation 9/450 queries; elapsed=37s; saved selections reused
BEA preparation 10/450 queries; elapsed=44s; saved selections reused
BEA preparation 11/450 queries; elapsed=51s; saved selections reused
BEA preparation 12/450 queries; elapsed=57s; saved selections reused
BEA preparation 13/450 queries; elapsed=62s; saved selections reused
BEA preparation 14/450 queries; elapsed=67s; saved selections reused
BEA preparation 15/450 queries; elapsed=72s; saved selections reused
BEA preparation 16/450 queries; elapsed=78s; saved selections reused
BEA preparation 17/450 queries; elapsed=85s; saved selections reused
BEA preparation 19/450 queries; elapsed=

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.87it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BEA expanded saved 305/2700 rows; elapsed=8s; method=QBGF_FULL
BEA expanded saved 310/2700 rows; elapsed=10s; method=FIXED_SENTENCE
BEA expanded saved 315/2700 rows; elapsed=12s; method=BEA_QBGF
BEA expanded saved 320/2700 rows; elapsed=16s; method=QBGF_FULL
BEA expanded saved 325/2700 rows; elapsed=22s; method=FIXED_SENTENCE
BEA expanded saved 330/2700 rows; elapsed=27s; method=BEA_QBGF
BEA expanded saved 335/2700 rows; elapsed=32s; method=QBGF_FULL
BEA expanded saved 340/2700 rows; elapsed=35s; method=FIXED_SENTENCE
BEA expanded saved 345/2700 rows; elapsed=38s; method=BEA_QBGF
BEA expanded saved 350/2700 rows; elapsed=41s; method=QBGF_FULL
BEA expanded saved 355/2700 rows; elapsed=44s; method=FIXED_SENTENCE
BEA expanded saved 360/2700 rows; elapsed=47s; method=BEA_QBGF
BEA expanded saved 365/2700 rows; elapsed=52s; method=QBGF_FULL
BEA expanded saved 370/2700 rows; elapsed=53s; method=FIXED_SENTENCE
BEA expanded saved 375/2700 rows; elapsed=54s; method=BEA_QBGF
BEA expanded saved 38

,label,dataset,budget,method,n_queries,n_groups,metric,quality,EM,accuracy,invalid_format,evidence_tokens,prompt_tokens,generated_tokens,physical_generation_calls_original_receipts,conceptual_lm_calls
0,POST_W12_EXPLORATORY,HotpotQA,256,BEA_QBGF,150,150,f1,0.335456,0.160000,NaN,NaN,218.546667,273.080000,15.266667,150,150
1,POST_W12_EXPLORATORY,HotpotQA,256,FIXED_SENTENCE,150,150,f1,0.410944,0.213333,NaN,NaN,254.340000,308.873333,13.546667,150,150
2,POST_W12_EXPLORATORY,HotpotQA,256,QBGF_FULL,150,150,f1,0.404153,0.193333,NaN,NaN,251.386667,305.920000,14.080000,148,150
3,POST_W12_EXPLORATORY,HotpotQA,1024,BEA_QBGF,150,150,f1,0.335193,0.166667,NaN,NaN,266.880000,321.413333,15.080000,80,150
4,POST_W12_EXPLORATORY,HotpotQA,1024,FIXED_SENTENCE,150,150,f1,0.419601,0.206667,NaN,NaN,994.440000,1048.973333,15.000000,149,150
5,POST_W12_EXPLORATORY,HotpotQA,1024,QBGF_FULL,150,150,f1,0.426004,0.213333,NaN,NaN,988.600000,1043.133333,14.853333,127,150
6,POST_W12_EXPLORATORY,QASPER,256,BEA_QBGF,150,120,f1,0.192534,0.013333,NaN,NaN,243.786667,286.640000,17.686667,150,150
7,POST_W12_EXPLORATORY,QASPER,256,FIXED_SENTENCE,150,120,f1,0.252850,0.020000,NaN,NaN,255.293333,298.146667,18.520000,150,150
8,POST_W12_EXPLORATORY,QASPER,256,QBGF_FULL,150,120,f1,0.242317,0.033333,NaN,NaN,252.446667,295.300000,18.406667,149,150
9,POST_W12_EXPLORATORY,QASPER,1024,BEA_QBGF,150,120,f1,0.239193,0.020000,NaN,NaN,477.193333,520.046667,18.306667,124,150


## 7. Optional attribution — disabled by default
Only after a compatible PROMISING pilot: 20 frozen pilot VAL examples per dataset, both budgets, BEA LOO and parent-group removal. New generation is BEA-only; historical baseline LOO is read-only and reused only where exact compatibility is provable. Missing control LOO is left unavailable. Zero effect is not useless evidence.

In [11]:
RUN_ATTRIBUTION = True
if RUN_ATTRIBUTION:
    pilot_analysis = analyze(cfg, stage='pilot')
    from src.post_w12.bea_qbgf.attribution import run as run_attribution
    display(run_attribution(cfg))
else:
    print('Optional LOO/group attribution not launched.')
print('Frozen W12 policy remains FIXED_SENTENCE. Outputs:', RESULT_ROOT)
if (RESULT_ROOT/'FINAL_POST_W12_REPORT.md').exists():
    display(Markdown((RESULT_ROOT/'FINAL_POST_W12_REPORT.md').read_text()))


Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.89it/s]


Optional BEA attribution saved HotpotQA 5a7741775542994aec3b724f
Optional BEA attribution saved HotpotQA 5a7949d45542994bb9456fdd
Optional BEA attribution saved HotpotQA 5a7c76275542990527d554b4
Optional BEA attribution saved HotpotQA 5a807e6f5542995d8a8ddf80
Optional BEA attribution saved HotpotQA 5a8484495542997175ce1ee8
Optional BEA attribution saved HotpotQA 5a8901a2554299515336125a
Optional BEA attribution saved HotpotQA 5a89ba6b554299669944a584
Optional BEA attribution saved HotpotQA 5a8dbc8b554299441c6ba077
Optional BEA attribution saved HotpotQA 5a8eed625542995085b374b9
Optional BEA attribution saved HotpotQA 5a8f7c9a55429918e830d239
Optional BEA attribution saved HotpotQA 5ab2f88755429976abd1bbe0
Optional BEA attribution saved HotpotQA 5ab7c6995542993667794005
Optional BEA attribution saved HotpotQA 5ab9b7d555429970cfb8eb7a
Optional BEA attribution saved HotpotQA 5abd8c295542992ac4f382ab
Optional BEA attribution saved HotpotQA 5adc75525542994650320d0e
Optional BEA attribution 

[{'dataset': 'HotpotQA',
  'budget': 256,
  'method': 'BEA_QBGF',
  'n_queries': 20,
  'n_units': 352,
  'neutral_unit_fraction': 0.8607954545454546,
  'neutral_token_fraction': 0.8306739699928555,
  'token_definition': 'Exact source-text Qwen tokens per exposed unit; excludes framing; not additive serialized budget',
  'coverage': 'Legacy baselines only where exact LOO cache exists; missing rows are not regenerated'},
 {'dataset': 'HotpotQA',
  'budget': 1024,
  'method': 'BEA_QBGF',
  'n_queries': 20,
  'n_units': 379,
  'neutral_unit_fraction': 0.9261213720316622,
  'neutral_token_fraction': 0.9039640375970576,
  'token_definition': 'Exact source-text Qwen tokens per exposed unit; excludes framing; not additive serialized budget',
  'coverage': 'Legacy baselines only where exact LOO cache exists; missing rows are not regenerated'},
 {'dataset': 'QASPER',
  'budget': 256,
  'method': 'BEA_QBGF',
  'n_queries': 20,
  'n_units': 388,
  'neutral_unit_fraction': 0.7216494845360825,
  'ne

Frozen W12 policy remains FIXED_SENTENCE. Outputs: /home/gunavenkat/Downloads/CS787-RAG-Project/results/post_w12/bea_qbgf/v3


# BEA-QBGF pilot — POST_W12_EXPLORATORY

Decision: **PROMISING**. Frozen Week12 remains SFR top10 → FIXED_SENTENCE → Qwen3-8B BASE_FP16. These are reused VAL examples, not fresh confirmation or OOD.

## All quality and cost cells

| label                | dataset   |   budget | method         |   n_queries |   n_groups | metric   |   quality |     EM |   accuracy |   invalid_format |   evidence_tokens |   prompt_tokens |   generated_tokens |   physical_generation_calls_original_receipts |   conceptual_lm_calls |
|:---------------------|:----------|---------:|:---------------|------------:|-----------:|:---------|----------:|-------:|-----------:|-----------------:|------------------:|----------------:|-------------------:|----------------------------------------------:|----------------------:|
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF       |          50 |         50 | f1       |  0.297642 |   0.18 |     nan    |              nan |            225.26 |          280.14 |              16.92 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | FIXED_SENTENCE |          50 |         50 | f1       |  0.366109 |   0.22 |     nan    |              nan |            254.84 |          309.72 |              13.54 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | QBGF_FULL      |          50 |         50 | f1       |  0.404225 |   0.28 |     nan    |              nan |            251.62 |          306.5  |              13.6  |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF       |          50 |         50 | f1       |  0.344117 |   0.22 |     nan    |              nan |            284.64 |          339.52 |              15.76 |                                            28 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | FIXED_SENTENCE |          50 |         50 | f1       |  0.389292 |   0.24 |     nan    |              nan |           1001.74 |         1056.62 |              15.34 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | QBGF_FULL      |          50 |         50 | f1       |  0.402047 |   0.26 |     nan    |              nan |            995.26 |         1050.14 |              15.28 |                                            44 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF       |          50 |         50 | f1       |  0.186849 |   0.02 |     nan    |              nan |            245.74 |          288.32 |              18.08 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |      256 | FIXED_SENTENCE |          50 |         50 | f1       |  0.266794 |   0.02 |     nan    |              nan |            255.06 |          297.64 |              18.26 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |      256 | QBGF_FULL      |          50 |         50 | f1       |  0.264119 |   0.02 |     nan    |              nan |            252.52 |          295.1  |              18.74 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF       |          50 |         50 | f1       |  0.208418 |   0.02 |     nan    |              nan |            472.78 |          515.36 |              19.22 |                                            42 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |     1024 | FIXED_SENTENCE |          50 |         50 | f1       |  0.289603 |   0.02 |     nan    |              nan |           1023.16 |         1065.74 |              20.42 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | QASPER    |     1024 | QBGF_FULL      |          50 |         50 | f1       |  0.296416 |   0.02 |     nan    |              nan |           1020.44 |         1063.02 |              19.44 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF       |          50 |         50 | macro_f1 |  0.612738 | nan    |       0.62 |                0 |            250.08 |          314.04 |               3.84 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |      256 | FIXED_SENTENCE |          50 |         50 | macro_f1 |  0.628915 | nan    |       0.64 |                0 |            252.62 |          316.58 |               3.52 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |      256 | QBGF_FULL      |          50 |         50 | macro_f1 |  0.530994 | nan    |       0.54 |                0 |            250.56 |          314.52 |               3.8  |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF       |          50 |         50 | macro_f1 |  0.643155 | nan    |       0.66 |                0 |            696.02 |          759.98 |               3.68 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | FIXED_SENTENCE |          50 |         50 | macro_f1 |  0.553333 | nan    |       0.64 |                0 |           1020.7  |         1084.66 |               3.16 |                                            50 |                    50 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | QBGF_FULL      |          50 |         50 | macro_f1 |  0.588105 | nan    |       0.64 |                0 |           1018.72 |         1082.68 |               3.36 |                                            50 |                    50 |

## Paired differences

| label                | dataset   |   budget | left     | right          |      point |     ci_low |      ci_high |      raw_p |   resamples |   seed |   n_queries |   n_clusters | metric   | test                                                                          |   exploratory_family_size |    holm_p |
|:---------------------|:----------|---------:|:---------|:---------------|-----------:|-----------:|-------------:|-----------:|------------:|-------:|------------:|-------------:|:---------|:------------------------------------------------------------------------------|--------------------------:|----------:|
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF | FIXED_SENTENCE | -0.0684668 | -0.181965  |  0.0400673   | 0.243876   |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF | QBGF_FULL      | -0.106583  | -0.219946  |  0.000584362 | 0.0708929  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.567143  |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF | FIXED_SENTENCE | -0.0451749 | -0.137005  |  0.0433362   | 0.340766   |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF | QBGF_FULL      | -0.05793   | -0.16153   |  0.041834    | 0.274673   |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF | FIXED_SENTENCE | -0.0799456 | -0.152321  | -0.0146743   | 0.0284972  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.284972  |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF | QBGF_FULL      | -0.0772708 | -0.15304   | -0.00739775  | 0.0493951  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.444556  |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF | FIXED_SENTENCE | -0.081185  | -0.149839  | -0.0155808   | 0.0218978  |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.240876  |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF | QBGF_FULL      | -0.0879982 | -0.151298  | -0.0301388   | 0.00429957 |       10000 |     42 |          50 |           50 | f1       | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 0.0515948 |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF | FIXED_SENTENCE | -0.0161764 | -0.181927  |  0.148248    | 0.829017   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF | QBGF_FULL      |  0.0817442 | -0.0662691 |  0.233638    | 0.292571   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF | FIXED_SENTENCE |  0.0898216 | -0.0542215 |  0.228745    | 0.293971   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF | QBGF_FULL      |  0.0550504 | -0.0727925 |  0.181957    | 0.429157   |       10000 |     42 |          50 |           50 | macro_f1 | two-sided paired cluster label-swap randomization; exchangeability under null |                        12 | 1         |

## Evidence behavior

| label                | dataset   |   budget | method         |   n_queries |   full_eligible_query |   full_considered |   full_chosen |   full_rejected_cost |   full_rejected_low_gain |   full_rejected_threshold |   full_lost_competition |   full_blocked_one_sentence |   evidence_tokens |   unused_budget |   budget_utilization |   selected_units |   represented_parents |   sentence_units |   full_units |   mean_pairwise_redundancy |   aspect_coverage |   represented_aspects |   tokens_per_represented_aspect |   fidelity_upgrades |   empty_evidence |   stopped_with_unused_budget |   pairwise_redundancy_defined_queries |   full_representation_frequency |   upgrade_cell_fraction |
|:---------------------|:----------|---------:|:---------------|------------:|----------------------:|------------------:|--------------:|---------------------:|-------------------------:|--------------------------:|------------------------:|----------------------------:|------------------:|----------------:|---------------------:|-----------------:|----------------------:|-----------------:|-------------:|---------------------------:|------------------:|----------------------:|--------------------------------:|--------------------:|-----------------:|-----------------------------:|--------------------------------------:|--------------------------------:|------------------------:|
| POST_W12_EXPLORATORY | HotpotQA  |      256 | BEA_QBGF       |          50 |                    50 |              1970 |            14 |                 1108 |                        8 |                       594 |                     246 |                        4532 |            225.26 |           30.74 |             0.879922 |            18.16 |                  9.66 |            17.88 |         0.28 |                   0.594611 |          0.748675 |                  1.64 |                         165.453 |                0.28 |                0 |                         0.62 |                                    50 |                            0.26 |                    0.26 |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            254.84 |            1.16 |             0.995469 |             9.8  |                  5.28 |             9.8  |         0    |                   0.650451 |          0.761499 |                  1.64 |                         187.278 |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | HotpotQA  |      256 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            251.62 |            4.38 |             0.982891 |            11.74 |                  4.94 |            11.74 |         0    |                   0.627003 |          0.759819 |                  1.64 |                         185.302 |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | BEA_QBGF       |          50 |                    50 |              2815 |            25 |                    0 |                        8 |                      2434 |                     348 |                        4802 |            284.64 |          739.36 |             0.277969 |            20.02 |                  9.72 |            19.52 |         0.5  |                   0.591976 |          0.755754 |                  1.64 |                         209.427 |                0.5  |                0 |                         1    |                                    50 |                            0.38 |                    0.38 |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1001.74 |           22.26 |             0.978262 |            36.06 |                  9.62 |            36.06 |         0    |                   0.585496 |          0.76449  |                  1.64 |                         737.265 |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | HotpotQA  |     1024 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            995.26 |           28.74 |             0.971934 |            38.28 |                  9.24 |            38.14 |         0.14 |                   0.573011 |          0.764431 |                  1.64 |                         732.538 |                0    |                0 |                         0    |                                    50 |                            0.12 |                    0    |
| POST_W12_EXPLORATORY | QASPER    |      256 | BEA_QBGF       |          50 |                    50 |              3027 |             0 |                 3013 |                        0 |                        12 |                       2 |                        3240 |            245.74 |           10.26 |             0.959922 |            20.3  |                  8.78 |            20.3  |         0    |                   0.58434  |          0.709078 |                  1.02 |                         243.23  |                0    |                0 |                         0.28 |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |      256 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            255.06 |            0.94 |             0.996328 |             9.34 |                  5.7  |             9.34 |         0    |                   0.686346 |          0.722814 |                  1.02 |                         252.51  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |      256 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            252.52 |            3.48 |             0.986406 |            10.22 |                  5.82 |            10.22 |         0    |                   0.68218  |          0.720867 |                  1.02 |                         249.99  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |     1024 | BEA_QBGF       |          50 |                    50 |              7785 |             0 |                  231 |                        0 |                      7027 |                     527 |                        3946 |            472.78 |          551.22 |             0.461699 |            31.9  |                  9.3  |            31.9  |         0    |                   0.587056 |          0.715636 |                  1.02 |                         468     |                0    |                0 |                         0.96 |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |     1024 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1023.16 |            0.84 |             0.99918  |            32.2  |                  9.62 |            32.2  |         0    |                   0.66966  |          0.722814 |                  1.02 |                        1012.92  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | QASPER    |     1024 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1020.44 |            3.56 |             0.996523 |            24.5  |                  8.56 |            23.1  |         1.4  |                   0.639274 |          0.722404 |                  1.02 |                        1010.26  |                0    |                0 |                         0    |                                    50 |                            0.86 |                    0    |
| POST_W12_EXPLORATORY | SciFact   |      256 | BEA_QBGF       |          50 |                    50 |              1462 |             0 |                 1401 |                        0 |                        24 |                      37 |                        2703 |            250.08 |            5.92 |             0.976875 |            15.46 |                  8.16 |            15.46 |         0    |                   0.629506 |          0.777207 |                  1.2  |                         225.12  |                0    |                0 |                         0.04 |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |      256 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            252.62 |            3.38 |             0.986797 |             6.56 |                  3.22 |             6.56 |         0    |                   0.750613 |          0.802011 |                  1.2  |                         227.44  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |      256 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |            250.56 |            5.44 |             0.97875  |             8.5  |                  3.36 |             8.5  |         0    |                   0.721572 |          0.785962 |                  1.2  |                         225.29  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |     1024 | BEA_QBGF       |          50 |                    50 |              8027 |             2 |                 1281 |                        0 |                      6490 |                     254 |                        4871 |            696.02 |          327.98 |             0.679707 |            33.98 |                  9.8  |            33.94 |         0.04 |                   0.634118 |          0.789252 |                  1.2  |                         624.31  |                0.04 |                0 |                         0.9  |                                    50 |                            0.04 |                    0.04 |
| POST_W12_EXPLORATORY | SciFact   |     1024 | FIXED_SENTENCE |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1020.7  |            3.3  |             0.996777 |            24.1  |                  8.3  |            24.1  |         0    |                   0.718169 |          0.804624 |                  1.2  |                         918.67  |                0    |                0 |                         0    |                                    50 |                            0    |                    0    |
| POST_W12_EXPLORATORY | SciFact   |     1024 | QBGF_FULL      |          50 |                     0 |                 0 |             0 |                    0 |                        0 |                         0 |                       0 |                           0 |           1018.72 |            5.28 |             0.994844 |            27.7  |                  7.02 |            27.22 |         0.48 |                   0.674211 |          0.798838 |                  1.2  |                         916.89  |                0    |                0 |                         0    |                                    50 |                            0.42 |                    0    |

## Questions answered

1. **Better than QBGF?** See BEA−QBGF paired differences for every dataset/budget; mixed cells do not establish universal superiority.

2. **Competitive with FIXED_SENTENCE?** See BEA−FIXED paired intervals. Point competitiveness in the pilot gate is not a formal noninferiority result.

3. **Less redundancy?** Compare mean_pairwise_redundancy with its defined-query counts; coverage of FULL uses its constituent sentences for comparability.

4. **More query-aspect coverage?** Compare aspect_coverage under identical query-only aspects. Cosine coverage is a surrogate, not verified fact coverage.

5. **Stops with unused tokens?** See unused_budget, stopped_with_unused_budget and saved per-step stop reasons.

6. **Fidelity escalation active?** Upgrade-cell fraction: 0.1133; inspect source accumulation and FULL actions in cache/selections.

7. **Does expansion itself improve answers?** Unresolved: frequency is measured, but there is no separate BEA-without-expansion primary arm. Do not infer causal benefit from activation or between-method scores.

8. **Gains at 256 or 1024?** The complete paired table reports both budgets; do not choose a budget after outcomes.

9. **Dataset dependence?** Report HotpotQA/QASPER answer F1 and SciFact aggregate macro-F1 separately; never pool them.

10. **Top-down relevance-mass hypothesis supported?** Mechanism diagnostics can support or weaken its plausibility, but without new LOO this remains unresolved as a claim about neutral downstream evidence. Relevance/coverage/redundancy proxies are not downstream utility labels.

## Scope and costs

All three methods share exact inputs, frozen serializer, reader, prompt/parser, precision and decoding. Physical cache reuse is recorded per row; every system still costs one conceptual answer call. Embedding lookup/physical/original preprocessing costs and query/aspect computation are recorded in cache/problems; they are distinct from generation. No end-to-end latency or serving-throughput claim is made. No gold/LOO/answer-score fields enter allocation. Novelty is diagnostic only, not a duplicate reward. FULL utility sums sequentially recomputed positive marginal sentence surrogates, updating redundancy and coverage after hypothetical additions; it remains a surrogate rather than learned downstream contextual utility.

Optional attribution is not run automatically. Neutral LOO means zero score change under removal, not useless or unobserved evidence. Expanded VAL is blocked unless the pilot passes and the user enables it; it is not independent confirmation. Historical W12 confirmation/final freeze remain unchanged.
Physical call totals in summary are original creation receipts, including pilot rows reused in expanded VAL. For physical calls in the latest invocation, use that stage’s cache_manifest.csv; reused rows are never presented as newly generated.
